# LeetCode 1438 — Longest Continuous Subarray With Absolute Diff Less Than or Equal to Limit bằng Python

Link đề: https://leetcode.com/problems/longest-continuous-subarray-with-absolute-diff-less-than-or-equal-to-limit/ · Độ khó: **Medium** · Chủ đề: Array, Queue, Sliding Window, Heap (Priority Queue), Ordered Set, Monotonic Queue

Skeleton LeetCode:

```python
class Solution:
    def longestSubarray(self, nums: list[int], limit: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra rằng điều kiện "mọi cặp phần tử chênh lệch ≤ `limit`" tương đương **`max − min ≤ limit`** — biến một điều kiện `O(k²)` cặp thành điều kiện chỉ cần 2 con số.
2. Viết được **brute force `O(n²)`** có giữ `min/max` chạy dần, và hiểu vì sao nó quá chậm với `n = 10^5`.
3. Nắm kỹ thuật **cửa sổ trượt hai con trỏ** (mở rộng phải, co trái) cho bài "đoạn con dài nhất thoả điều kiện đơn điệu".
4. Duy trì `max/min` của cửa sổ bằng 3 cấu trúc: **danh sách có thứ tự (`bisect`)**, **hai heap xoá lười**, và **hai deque đơn điệu** (`O(n)`).
5. Tránh các lỗi kinh điển: chỉ so phần tử kề nhau, deque làm mất phần tử trùng, lệch một đơn vị khi tính độ dài.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `limit`. Hãy trả về **độ dài của đoạn con liên tiếp (subarray) không rỗng dài nhất** sao cho **hiệu tuyệt đối giữa mọi cặp phần tử** trong đoạn đó đều **≤ `limit`**.

Chú ý: "mọi cặp" — không chỉ các phần tử kề nhau.

**Ví dụ 1:** `nums = [8,2,4,7], limit = 4` → `2`. Các đoạn hợp lệ dài nhất là `[2,4]` hoặc `[4,7]`. Đoạn `[2,4,7]` sai vì cặp `(2,7)` chênh 5 > 4 dù 2 và 7 không đứng cạnh nhau.

**Ví dụ 2:** `nums = [10,1,2,4,7,2], limit = 5` → `4`. Đoạn `[2,4,7,2]` có chênh lệch lớn nhất là `|7−2| = 5`.

**Ví dụ 3:** `nums = [4,2,2,2,4,4,2,2], limit = 0` → `3`. Với `limit = 0` mọi phần tử phải bằng nhau, đoạn dài nhất là `[2,2,2]`.

**Ràng buộc:**
- `1 <= nums.length <= 10^5`
- `1 <= nums[i] <= 10^9`
- `0 <= limit <= 10^9`

In [1]:
import random
import time
import heapq
from bisect import bisect_left, insort
from collections import deque
from typing import List

examples = [
    (([8, 2, 4, 7], 4), 2),
    (([10, 1, 2, 4, 7, 2], 5), 4),
    (([4, 2, 2, 2, 4, 4, 2, 2], 0), 3),
]
for (nums, limit), want in examples:
    print(f"nums={nums}, limit={limit}  ->  kỳ vọng {want}")

nums=[8, 2, 4, 7], limit=4  ->  kỳ vọng 2
nums=[10, 1, 2, 4, 7, 2], limit=5  ->  kỳ vọng 4
nums=[4, 2, 2, 2, 4, 4, 2, 2], limit=0  ->  kỳ vọng 3


## 3. Kiến thức nền

### 3.1 Subarray (đoạn con liên tiếp) và slicing

Subarray là một khúc **liên tiếp** `nums[i..j]`. Trong Python viết `nums[i:j+1]` (cận phải không lấy). Một mảng dài `n` có `n·(n+1)/2` subarray — với `n = 10^5` là khoảng 5·10^9, không thể duyệt hết.

In [2]:
nums = [8, 2, 4, 7]
n = len(nums)
subs = [nums[i:j + 1] for i in range(n) for j in range(i, n)]
print(len(subs), "=", n * (n + 1) // 2)
print(subs)

10 = 10
[[8], [8, 2], [8, 2, 4], [8, 2, 4, 7], [2], [2, 4], [2, 4, 7], [4], [4, 7], [7]]


### 3.2 "Mọi cặp chênh ≤ limit" ⇔ `max − min ≤ limit`

Cặp chênh lệch lớn nhất trong một tập luôn là cặp `(max, min)`. Nếu cặp đó đã ≤ `limit` thì mọi cặp khác cũng vậy. Do đó chỉ cần theo dõi **2 giá trị** của đoạn: lớn nhất và nhỏ nhất.

In [3]:
from itertools import combinations
seg = [2, 4, 7]
pairs = [(a, b, abs(a - b)) for a, b in combinations(seg, 2)]
print(pairs)
print("max chênh theo cặp:", max(d for _, _, d in pairs), "| max - min:", max(seg) - min(seg))

[(2, 4, 2), (2, 7, 5), (4, 7, 3)]
max chênh theo cặp: 5 | max - min: 5


### 3.3 `collections.deque` — hàng đợi hai đầu

`deque` cho phép `append/pop` ở **cuối** và `appendleft/popleft` ở **đầu** đều `O(1)`. Truy cập `dq[0]` (đầu) và `dq[-1]` (cuối) cũng `O(1)`. List thường thì `pop(0)` tốn `O(n)`.

In [4]:
dq = deque()
for x in [5, 3, 8]:
    dq.append(x)
print(dq, "đầu =", dq[0], "cuối =", dq[-1])
dq.popleft(); dq.pop()
print(dq)

deque([5, 3, 8]) đầu = 5 cuối = 8
deque([3])


### 3.4 `heapq` — min-heap, và mẹo đổi dấu để có max-heap

`heapq` luôn giữ phần tử **nhỏ nhất** ở `h[0]`. Muốn max-heap thì đẩy `-x`. Ta có thể đẩy tuple `(giá trị, chỉ số)` để biết phần tử nằm ở đâu trong mảng — rất cần cho kỹ thuật **xoá lười** (lazy deletion).

In [5]:
mn, mx = [], []
for i, x in enumerate([8, 2, 4, 7]):
    heapq.heappush(mn, (x, i))
    heapq.heappush(mx, (-x, i))
print("min:", mn[0], "| max:", (-mx[0][0], mx[0][1]))

min: (2, 1) | max: (8, 0)


### 3.5 `bisect.insort` và `bisect_left` — giữ một list luôn được sắp xếp

`insort(a, x)` chèn `x` vào đúng chỗ để `a` vẫn tăng dần; `bisect_left(a, x)` tìm vị trí đầu tiên `>= x` (dùng để xoá một giá trị). Khi đó `a[0]` là min, `a[-1]` là max. Tìm vị trí là `O(log k)` nhưng chèn/xoá phải dịch phần tử nên `O(k)` — trong thực tế rất nhanh vì dùng `memmove`.

In [6]:
a = []
for x in [8, 2, 4, 7, 4]:
    insort(a, x)
print(a, "min =", a[0], "max =", a[-1])
a.pop(bisect_left(a, 4))   # xoá MỘT bản sao của 4
print(a)

[2, 4, 4, 7, 8] min = 2 max = 8
[2, 4, 7, 8]


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2: `nums = [10,1,2,4,7,2]`, `limit = 5`. Với mỗi điểm bắt đầu `i`, ta kéo dài sang phải và theo dõi `min`, `max`:

| i | đoạn kéo dài | min | max | max−min | hợp lệ? |
|---|---|---|---|---|---|
| 0 | [10] | 10 | 10 | 0 | ✔ |
| 0 | [10,1] | 1 | 10 | 9 | ✘ → dừng |
| 1 | [1,2,4] | 1 | 4 | 3 | ✔ |
| 1 | [1,2,4,7] | 1 | 7 | 6 | ✘ → dừng (dài nhất từ 1 là 3) |
| 2 | [2,4,7,2] | 2 | 7 | 5 | ✔ (dài 4) |

Hai quan sát quan trọng:
1. Khi đoạn `[i..j]` đã **sai** (max−min > limit) thì kéo dài thêm cũng **sai** (max chỉ tăng, min chỉ giảm). ⇒ có thể **dừng sớm**.
2. Nếu `[i..j]` **đúng** thì mọi đoạn con bên trong nó cũng **đúng**. ⇒ khi tăng `i`, điểm cuối tốt nhất `j` **không lùi lại**. Đây chính là mầm mống của cửa sổ trượt.

In [7]:
nums, limit = [10, 1, 2, 4, 7, 2], 5
for i in range(len(nums)):
    lo = hi = nums[i]
    j = i
    while j + 1 < len(nums) and max(hi, nums[j + 1]) - min(lo, nums[j + 1]) <= limit:
        j += 1
        lo, hi = min(lo, nums[j]), max(hi, nums[j])
    print(f"i={i}: đoạn dài nhất bắt đầu tại i là {nums[i:j+1]} (độ dài {j-i+1})")

i=0: đoạn dài nhất bắt đầu tại i là [10] (độ dài 1)
i=1: đoạn dài nhất bắt đầu tại i là [1, 2, 4] (độ dài 3)
i=2: đoạn dài nhất bắt đầu tại i là [2, 4, 7, 2] (độ dài 4)
i=3: đoạn dài nhất bắt đầu tại i là [4, 7, 2] (độ dài 3)
i=4: đoạn dài nhất bắt đầu tại i là [7, 2] (độ dài 2)
i=5: đoạn dài nhất bắt đầu tại i là [2] (độ dài 1)


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử mọi điểm bắt đầu `i`; từ `i` kéo `j` sang phải, cập nhật `lo = min(lo, nums[j])`, `hi = max(hi, nums[j])` trong `O(1)`. Ngay khi `hi − lo > limit` thì `break` (quan sát 1).

Nếu làm ngây thơ hơn — tính `max(nums[i:j+1]) - min(...)` cho từng cặp `(i, j)` — sẽ là `O(n³)`. Giữ `min/max` chạy dần giúp giảm còn `O(n²)`.

In [8]:
def longest_brute(nums: List[int], limit: int) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        lo = hi = nums[i]
        for j in range(i, n):
            lo = min(lo, nums[j])
            hi = max(hi, nums[j])
            if hi - lo > limit:
                break               # kéo dài thêm cũng vô ích
            best = max(best, j - i + 1)
    return best

for (nums, limit), want in examples:
    print(longest_brute(nums, limit), "kỳ vọng", want)

2 kỳ vọng 2
4 kỳ vọng 4
3 kỳ vọng 3


**Độ phức tạp:** Time `O(n²)` trong trường hợp xấu (ví dụ `limit` rất lớn, không bao giờ `break`), Space `O(1)`.

**Nút thắt:** với `n = 10^5` và `limit = 10^9`, vòng trong chạy đủ `~5·10^9` bước → TLE. Ta lãng phí vì mỗi lần tăng `i` lại bắt đầu `j` từ đầu, trong khi quan sát 2 nói `j` không cần lùi.

In [9]:
# Đo thời gian brute force trên trường hợp xấu (limit lớn) với n vừa phải
nums_bad = list(range(1, 3001))
t = time.perf_counter()
print(longest_brute(nums_bad, 10**9))
print(f"n=3000 mất {time.perf_counter()-t:.2f}s — n=10^5 sẽ chậm hơn ~1000 lần")

3000
n=3000 mất 0.52s — n=10^5 sẽ chậm hơn ~1000 lần


## 6. Ý tưởng thuật toán / bước đột phá

**Insight — cửa sổ trượt (two pointers):** duy trì cửa sổ `[l..r]` luôn **hợp lệ**.
- Mỗi bước, **mở rộng phải**: thêm `nums[r]` vào cửa sổ.
- **Nếu** `max − min > limit` **thì** co trái (`l += 1`, bỏ `nums[l]` khỏi cửa sổ) **cho đến khi** hợp lệ lại.
- Cập nhật `best = max(best, r − l + 1)`.

**Invariant:** sau khi xử lý xong `r`, `[l..r]` là đoạn hợp lệ **dài nhất kết thúc tại `r`**. Vì `l` và `r` chỉ tăng, tổng số bước di chuyển là `≤ 2n`.

**Vấn đề còn lại:** làm sao biết `max` và `min` của cửa sổ thật nhanh khi phần tử vừa được thêm bên phải vừa bị bỏ bên trái? Có 3 lựa chọn, từ dễ hiểu đến tối ưu:

| Cấu trúc | Thêm | Bỏ trái | Lấy max/min |
|---|---|---|---|
| list có thứ tự + `bisect` | `O(k)` (dịch phần tử) | `O(k)` | `O(1)` |
| 2 heap + xoá lười | `O(log n)` | lười (chỉ bỏ khi ở đỉnh) | `O(log n)` khấu hao |
| 2 deque đơn điệu | `O(1)` khấu hao | `O(1)` | `O(1)` |

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cửa sổ trượt + list có thứ tự (`bisect`)

**Giải thích từng đoạn:**
1. `window` là list **đã sắp xếp** chứa đúng các phần tử `nums[l..r]`.
2. Thêm `nums[r]` bằng `insort` (giữ thứ tự).
3. `window[-1] - window[0]` chính là `max − min`. Nếu > limit: xoá `nums[l]` bằng `bisect_left` + `pop`, rồi `l += 1`.
4. Đây là phiên bản Python của "multiset" mà gợi ý của đề nhắc đến (C++ `std::multiset`).

In [10]:
def longest_sorted(nums: List[int], limit: int) -> int:
    window = []          # các phần tử nums[l..r], luôn tăng dần
    l = best = 0
    for r, x in enumerate(nums):
        insort(window, x)
        while window[-1] - window[0] > limit:
            window.pop(bisect_left(window, nums[l]))
            l += 1
        best = max(best, r - l + 1)
    return best

for (nums, limit), want in examples:
    print(longest_sorted(nums, limit), "kỳ vọng", want)

2 kỳ vọng 2
4 kỳ vọng 4
3 kỳ vọng 3


**Code nộp LeetCode (cách 2):**

In [11]:
class SolutionSorted:
    def longestSubarray(self, nums: List[int], limit: int) -> int:
        window, l, best = [], 0, 0
        for r, x in enumerate(nums):
            insort(window, x)
            while window[-1] - window[0] > limit:
                window.pop(bisect_left(window, nums[l]))
                l += 1
            best = max(best, r - l + 1)
        return best

print(SolutionSorted().longestSubarray([10, 1, 2, 4, 7, 2], 5))

4


**Độ phức tạp:** Time `O(n·k)` với `k` là kích thước cửa sổ (xấu nhất `O(n²)` về lý thuyết, nhưng việc dịch phần tử dùng `memmove` nên thực tế vẫn qua được). Space `O(k)`. Nếu được dùng `sortedcontainers.SortedList` (LeetCode có sẵn) thì mỗi thao tác ~`O(log n)`.

### 7.2 Cách 3 — Cửa sổ trượt + hai heap xoá lười

**Giải thích từng đoạn:**
1. `maxh` chứa `(-giá trị, chỉ số)`, `minh` chứa `(giá trị, chỉ số)`.
2. Không thể xoá phần tử ở giữa heap, nên ta **xoá lười**: phần tử có chỉ số `< l` đã rời cửa sổ; chỉ khi nó **nổi lên đỉnh** ta mới `heappop`.
3. Khi `max − min > limit`: tăng `l` lên **qua khỏi** phần tử gây vi phạm — cụ thể đặt `l = min(chỉ số đỉnh max, chỉ số đỉnh min) + 1`. Mọi phần tử trước đó đều phải rời đi vì cửa sổ không thể chứa đồng thời cả max lẫn min hiện tại.
4. Sau đó dọn các đỉnh có chỉ số `< l`, và lặp lại kiểm tra.

In [12]:
def longest_heap(nums: List[int], limit: int) -> int:
    maxh, minh = [], []
    l = best = 0
    for r, x in enumerate(nums):
        heapq.heappush(maxh, (-x, r))
        heapq.heappush(minh, (x, r))
        while -maxh[0][0] - minh[0][0] > limit:
            l = min(maxh[0][1], minh[0][1]) + 1   # nhảy qua phần tử gây vi phạm
            while maxh[0][1] < l:                 # xoá lười
                heapq.heappop(maxh)
            while minh[0][1] < l:
                heapq.heappop(minh)
        best = max(best, r - l + 1)
    return best

for (nums, limit), want in examples:
    print(longest_heap(nums, limit), "kỳ vọng", want)

2 kỳ vọng 2
4 kỳ vọng 4
3 kỳ vọng 3


Vì sao `while maxh[0][1] < l` không bao giờ làm heap rỗng? Vì phần tử `r` vừa được đẩy vào có chỉ số `r ≥ l`, nên nó luôn còn trong cả hai heap.

**Code nộp LeetCode (cách 3):**

In [13]:
class SolutionHeap:
    def longestSubarray(self, nums: List[int], limit: int) -> int:
        maxh, minh = [], []
        l = best = 0
        for r, x in enumerate(nums):
            heapq.heappush(maxh, (-x, r))
            heapq.heappush(minh, (x, r))
            while -maxh[0][0] - minh[0][0] > limit:
                l = min(maxh[0][1], minh[0][1]) + 1
                while maxh[0][1] < l:
                    heapq.heappop(maxh)
                while minh[0][1] < l:
                    heapq.heappop(minh)
            best = max(best, r - l + 1)
        return best

print(SolutionHeap().longestSubarray([4, 2, 2, 2, 4, 4, 2, 2], 0))

3


**Độ phức tạp:** mỗi phần tử được push 1 lần và pop tối đa 1 lần vào mỗi heap ⇒ Time `O(n log n)`, Space `O(n)` (heap có thể giữ phần tử "đã chết" chưa bị dọn).

### 7.3 Cách 4 — Cửa sổ trượt + hai deque đơn điệu (tối ưu `O(n)`)

**Ý tưởng deque đơn điệu cho max:** giữ `maxq` chứa **chỉ số**, sao cho giá trị tương ứng **giảm dần** từ đầu đến cuối. Khi thêm `nums[r]`, mọi phần tử ở cuối có giá trị **< `nums[r]`** sẽ **không bao giờ** là max nữa (vì `nums[r]` lớn hơn và rời cửa sổ muộn hơn) ⇒ bỏ chúng đi. Đầu deque luôn là chỉ số của max cửa sổ.

Tương tự `minq` giữ giá trị **tăng dần**, đầu là min.

**Giải thích từng đoạn:**
1. Thêm `r`: `while maxq and nums[maxq[-1]] < x: maxq.pop()` rồi `maxq.append(r)`; tương tự cho `minq` với `>`.
2. Nếu `nums[maxq[0]] - nums[minq[0]] > limit`: tăng `l`; nếu đầu deque có chỉ số `< l` thì `popleft`.
3. Cập nhật `best`.

Lưu chỉ số (không lưu giá trị) giúp xử lý phần tử trùng nhau an toàn — xem lỗi số 3 ở mục 13.

In [14]:
def longest_deque(nums: List[int], limit: int) -> int:
    maxq, minq = deque(), deque()   # chỉ số; giá trị giảm dần / tăng dần
    l = best = 0
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] < x:
            maxq.pop()
        maxq.append(r)
        while minq and nums[minq[-1]] > x:
            minq.pop()
        minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > limit:
            l += 1
            if maxq[0] < l:
                maxq.popleft()
            if minq[0] < l:
                minq.popleft()
        best = max(best, r - l + 1)
    return best

for (nums, limit), want in examples:
    print(longest_deque(nums, limit), "kỳ vọng", want)

2 kỳ vọng 2
4 kỳ vọng 4
3 kỳ vọng 3


**Code nộp LeetCode (cách 4)** — xem mục 9 (đây là lời giải chính). **Độ phức tạp:** mỗi chỉ số được `append` 1 lần và bị `pop/popleft` tối đa 1 lần ở mỗi deque; `l` tăng tối đa `n` lần ⇒ Time `O(n)`, Space `O(n)`.

So sánh tốc độ thực tế trên `n = 10^5` (bỏ qua brute force):

In [15]:
rng = random.Random(2024)
big = [rng.randint(1, 10**9) for _ in range(10**5)]
for name, f in [("sorted/bisect", longest_sorted), ("heap", longest_heap), ("deque", longest_deque)]:
    t = time.perf_counter()
    res = f(big, 3 * 10**8)
    print(f"{name:14} -> {res:6d}  ({time.perf_counter()-t:.3f}s)")

sorted/bisect  ->     11  (0.023s)
heap           ->     11  (0.066s)
deque          ->     11  (0.036s)


## 8. TRACE từng bước

Trace cách 4 (deque) trên ví dụ 2: `nums = [10,1,2,4,7,2]`, `limit = 5`. Deque in ra dưới dạng **giá trị** cho dễ đọc (trong code là chỉ số).

| r | x | maxq (giá trị) | minq (giá trị) | max−min | co trái? | l | best |
|---|---|---|---|---|---|---|---|
| 0 | 10 | [10] | [10] | 0 | – | 0 | 1 |
| 1 | 1 | [10,1] | [1] | 9 | l→1, bỏ 10 | 1 | 1 |
| 2 | 2 | [2] | [1,2] | 1 | – | 1 | 2 |
| 3 | 4 | [4] | [1,2,4] | 3 | – | 1 | 3 |
| 4 | 7 | [7] | [1,2,4,7] | 6 | l→2, bỏ 1 | 2 | 3 |
| 5 | 2 | [7,2] | [2,2] | 5 | – | 2 | 4 |

In [16]:
def trace_deque(nums, limit):
    maxq, minq = deque(), deque()
    l = best = 0
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] < x:
            maxq.pop()
        maxq.append(r)
        while minq and nums[minq[-1]] > x:
            minq.pop()
        minq.append(r)
        shrink = []
        while nums[maxq[0]] - nums[minq[0]] > limit:
            shrink.append(nums[l])
            l += 1
            if maxq[0] < l:
                maxq.popleft()
            if minq[0] < l:
                minq.popleft()
        best = max(best, r - l + 1)
        print(f"r={r} x={x:2d} | maxq={[nums[i] for i in maxq]} minq={[nums[i] for i in minq]}"
              f" | bỏ trái {shrink} | l={l} cửa sổ={nums[l:r+1]} best={best}")
    return best

trace_deque([10, 1, 2, 4, 7, 2], 5)

r=0 x=10 | maxq=[10] minq=[10] | bỏ trái [] | l=0 cửa sổ=[10] best=1
r=1 x= 1 | maxq=[1] minq=[1] | bỏ trái [10] | l=1 cửa sổ=[1] best=1
r=2 x= 2 | maxq=[2] minq=[1, 2] | bỏ trái [] | l=1 cửa sổ=[1, 2] best=2
r=3 x= 4 | maxq=[4] minq=[1, 2, 4] | bỏ trái [] | l=1 cửa sổ=[1, 2, 4] best=3
r=4 x= 7 | maxq=[7] minq=[2, 4, 7] | bỏ trái [1] | l=2 cửa sổ=[2, 4, 7] best=3
r=5 x= 2 | maxq=[7, 2] minq=[2, 2] | bỏ trái [] | l=2 cửa sổ=[2, 4, 7, 2] best=4


4

Trace thêm cách 3 (heap) trên ví dụ 1 để thấy cú "nhảy" của `l` và việc xoá lười:

In [17]:
def trace_heap(nums, limit):
    maxh, minh = [], []
    l = best = 0
    for r, x in enumerate(nums):
        heapq.heappush(maxh, (-x, r))
        heapq.heappush(minh, (x, r))
        while -maxh[0][0] - minh[0][0] > limit:
            l = min(maxh[0][1], minh[0][1]) + 1
            while maxh[0][1] < l:
                heapq.heappop(maxh)
            while minh[0][1] < l:
                heapq.heappop(minh)
        best = max(best, r - l + 1)
        print(f"r={r} x={x} | đỉnh max={-maxh[0][0]}@{maxh[0][1]} đỉnh min={minh[0][0]}@{minh[0][1]}"
              f" | l={l} best={best} | kích thước heap: {len(maxh)},{len(minh)}")
    return best

trace_heap([8, 2, 4, 7], 4)

r=0 x=8 | đỉnh max=8@0 đỉnh min=8@0 | l=0 best=1 | kích thước heap: 1,1
r=1 x=2 | đỉnh max=2@1 đỉnh min=2@1 | l=1 best=1 | kích thước heap: 1,2
r=2 x=4 | đỉnh max=4@2 đỉnh min=2@1 | l=1 best=2 | kích thước heap: 2,3
r=3 x=7 | đỉnh max=7@3 đỉnh min=4@2 | l=2 best=2 | kích thước heap: 3,3


2

## 9. Code nộp LeetCode

Lời giải chính: cửa sổ trượt + hai deque đơn điệu, đúng chữ ký `longestSubarray(self, nums, limit)`.

In [18]:
from collections import deque
from typing import List

class Solution:
    def longestSubarray(self, nums: List[int], limit: int) -> int:
        maxq, minq = deque(), deque()
        l = best = 0
        for r, x in enumerate(nums):
            while maxq and nums[maxq[-1]] < x:
                maxq.pop()
            maxq.append(r)
            while minq and nums[minq[-1]] > x:
                minq.pop()
            minq.append(r)
            while nums[maxq[0]] - nums[minq[0]] > limit:
                l += 1
                if maxq[0] < l:
                    maxq.popleft()
                if minq[0] < l:
                    minq.popleft()
            best = max(best, r - l + 1)
        return best

sol = Solution()
for (nums, limit), want in examples:
    assert sol.longestSubarray(nums, limit) == want
print("Solution: 3 ví dụ đề OK")

Solution: 3 ví dụ đề OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đơn điệu):** nếu `[i..j]` hợp lệ thì mọi đoạn con `[i'..j']` với `i ≤ i' ≤ j' ≤ j` cũng hợp lệ, vì max chỉ giảm và min chỉ tăng khi bỏ bớt phần tử.

**Bổ đề 2 (con trỏ trái không lùi):** gọi `L(r)` là chỉ số nhỏ nhất sao cho `[L(r)..r]` hợp lệ. Khi đó `L(r) ≤ L(r+1)`: nếu `[L(r+1)..r+1]` hợp lệ thì theo bổ đề 1 `[L(r+1)..r]` cũng hợp lệ, nên `L(r) ≤ L(r+1)`.

**Invariant của vòng lặp:** sau khi xử lý `r`, biến `l = L(r)`. Vòng `while` chỉ tăng `l` khi cửa sổ còn sai, và dừng ngay khi đúng ⇒ `l` là chỉ số nhỏ nhất hợp lệ (vì `l` cũ = `L(r-1) ≤ L(r)` không bỏ sót). Đáp án là `max_r (r − L(r) + 1)` — đúng thứ `best` tính.

**Invariant của deque max:** các chỉ số trong `maxq` tăng dần, giá trị tương ứng giảm dần (không tăng), và chứa mọi "ứng viên max" của cửa sổ — tức mọi chỉ số `i ∈ [l..r]` sao cho không có `j > i` trong cửa sổ với `nums[j] > nums[i]`. Phần tử bị loại ở cuối có một phần tử lớn hơn đứng sau nó nên vĩnh viễn không thể là max. Do đó `nums[maxq[0]]` là max cửa sổ. Lập luận đối xứng cho `minq`.

**Cách heap:** khi `max − min > limit`, cửa sổ hợp lệ không thể chứa cả chỉ số của max lẫn min, nên phải bỏ đến phần tử đứng trước trong hai phần tử đó; nhảy `l = min(idx) + 1` chỉ bỏ những gì bắt buộc phải bỏ.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `k` = độ dài cửa sổ lớn nhất.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n²)` xấu nhất | `O(1)` | `break` sớm giúp nhanh khi `limit` nhỏ |
| 2. Sorted list + bisect | `O(n·k)` xấu nhất | `O(k)` | thực tế nhanh nhờ `memmove` |
| 3. Hai heap xoá lười | `O(n log n)` | `O(n)` | heap có thể giữ phần tử đã chết |
| 4. Hai deque đơn điệu | `O(n)` khấu hao | `O(n)` | mỗi chỉ số vào/ra deque tối đa 1 lần |

Phân tích khấu hao của cách 4: tổng số lệnh `pop` + `popleft` trên mỗi deque ≤ số lệnh `append` = `n`; vòng co trái chạy tổng cộng ≤ `n` lần.

## 12. Bộ test

Gồm: ví dụ đề, edge case (1 phần tử, `limit = 0`, `limit` cực lớn, toàn bằng nhau, tăng dần, giảm dần, xen kẽ), và đối chiếu ngẫu nhiên 4 cách với seed cố định.

In [19]:
ALL = {
    "brute": longest_brute,
    "sorted": longest_sorted,
    "heap": longest_heap,
    "deque": longest_deque,
    "Solution": lambda a, b: Solution().longestSubarray(a, b),
}

edge = [
    (([5], 0), 1),
    (([7, 7, 7, 7], 0), 4),
    (([1, 2, 3, 4, 5], 0), 1),
    (([1, 2, 3, 4, 5], 10**9), 5),
    (([1, 2, 3, 4, 5], 2), 3),
    (([5, 4, 3, 2, 1], 1), 2),
    (([1, 10, 1, 10, 1], 8), 1),
    (([1, 10, 1, 10, 1], 9), 5),
    (([1, 1, 2, 2], 0), 2),
    (([8, 2, 4, 7], 4), 2),
]
for (nums, limit), want in examples + edge:
    for name, f in ALL.items():
        got = f(list(nums), limit)
        assert got == want, (name, nums, limit, got, want)
print("Ví dụ đề + edge case: OK cho cả", len(ALL), "cách")

Ví dụ đề + edge case: OK cho cả 5 cách


In [20]:
rng = random.Random(1438)
for _ in range(1500):
    n = rng.randint(1, 30)
    hi = rng.choice([3, 10, 10**9])
    nums = [rng.randint(1, hi) for _ in range(n)]
    limit = rng.randint(0, hi)
    want = longest_brute(nums, limit)
    for name, f in ALL.items():
        assert f(nums, limit) == want, (name, nums, limit)
print("1500 test ngẫu nhiên (seed 1438): mọi cách khớp brute force")

1500 test ngẫu nhiên (seed 1438): mọi cách khớp brute force


In [21]:
# Test cỡ lớn đúng ràng buộc: n = 10^5, so sánh 3 cách nhanh với nhau
rng = random.Random(7)
for limit in [0, 5, 10**6, 10**9]:
    nums = [rng.randint(1, 10**9) if limit > 10 else rng.randint(1, 8) for _ in range(10**5)]
    a, b, c = longest_sorted(nums, limit), longest_heap(nums, limit), longest_deque(nums, limit)
    assert a == b == c
    print(f"limit={limit:>10}: {c}")

limit=         0: 6


limit=         5: 39


limit=   1000000: 2


limit=1000000000: 100000


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ so các phần tử **kề nhau**

Đề nói "mọi cặp", không phải "các cặp kề nhau". Code sai kiểm tra `|nums[i] − nums[i−1]| ≤ limit`.

In [22]:
def wrong_adjacent(nums, limit):
    best = cur = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if abs(nums[i] - nums[i - 1]) <= limit else 1
        best = max(best, cur)
    return best

print("Sai :", wrong_adjacent([8, 2, 4, 7], 4), "| Đúng:", longest_deque([8, 2, 4, 7], 4))
print("Vì [2,4,7] có các cặp kề chênh 2 và 3, nhưng cặp (2,7) chênh 5 > 4.")

Sai : 3 | Đúng: 2
Vì [2,4,7] có các cặp kề chênh 2 và 3, nhưng cặp (2,7) chênh 5 > 4.


**Sửa:** theo dõi `max − min` của cả đoạn (bất kỳ cách nào ở trên).

### Lỗi 2 — Lệch một đơn vị khi tính độ dài cửa sổ

Cửa sổ `[l..r]` (cả hai đầu đều tính) có độ dài `r − l + 1`, không phải `r − l`.

In [23]:
def wrong_length(nums, limit):
    maxq, minq = deque(), deque()
    l = best = 0
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] < x: maxq.pop()
        maxq.append(r)
        while minq and nums[minq[-1]] > x: minq.pop()
        minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > limit:
            l += 1
            if maxq[0] < l: maxq.popleft()
            if minq[0] < l: minq.popleft()
        best = max(best, r - l)        # SAI
    return best

print("Sai :", wrong_length([5], 0), "| Đúng:", longest_deque([5], 0))
print("Sai :", wrong_length([10, 1, 2, 4, 7, 2], 5), "| Đúng:", longest_deque([10, 1, 2, 4, 7, 2], 5))

Sai : 0 | Đúng: 1
Sai : 3 | Đúng: 4


**Sửa:** `best = max(best, r - l + 1)`.

### Lỗi 3 — Deque lưu **giá trị** và loại phần tử **bằng nhau** (`<=`)

Nhiều bạn lưu giá trị thay vì chỉ số, rồi khi co trái so `nums[l] == dq[0]` để quyết định `popleft`. Nếu lúc thêm lại pop cả phần tử **bằng** (`<=`), các bản sao bị mất: deque chỉ giữ một bản đại diện cho bản **mới nhất**, nhưng khi bản **cũ** rời cửa sổ ta lại `popleft` nhầm nó. Kết quả có thể sai, thậm chí deque rỗng gây `IndexError`. Ta bọc `try/except` để thấy hiện tượng mà không làm dừng notebook.

In [24]:
def wrong_values_deque(nums, limit):
    mx, mn = deque(), deque()
    l = best = 0
    for r, x in enumerate(nums):
        while mx and mx[-1] <= x:   # SAI: làm mất bản sao bằng nhau
            mx.pop()
        mx.append(x)
        while mn and mn[-1] >= x:   # SAI
            mn.pop()
        mn.append(x)
        while mx[0] - mn[0] > limit:
            if nums[l] == mx[0]: mx.popleft()
            if nums[l] == mn[0]: mn.popleft()
            l += 1
        best = max(best, r - l + 1)
    return best

for nums, limit in [([1, 1, 2, 2], 0), ([1, 1, 3, 3], 0)]:
    try:
        got = wrong_values_deque(nums, limit)
    except IndexError as e:
        got = f"IndexError({e})"
    print(f"{nums}, limit={limit}: Sai -> {got} | Đúng -> {longest_deque(nums, limit)}")

[1, 1, 2, 2], limit=0: Sai -> 3 | Đúng -> 2
[1, 1, 3, 3], limit=0: Sai -> 3 | Đúng -> 2


Với `[1,1,2,2]`, `limit=0`: khi thêm `1` thứ hai, bản `1` đầu bị pop; lúc thêm `2`, ta co trái bỏ `nums[0]=1` và `popleft` luôn bản `1` còn lại (vốn thuộc chỉ số 1) ⇒ min "biến mất", cửa sổ `[1,2]` sai lại được coi là hợp lệ.

**Sửa:** lưu **chỉ số** (như cách 4), hoặc nếu lưu giá trị thì chỉ pop phần tử **nhỏ hơn hẳn** (`<` cho maxq, `>` cho minq) để giữ lại các bản sao.

In [25]:
def fixed_values_deque(nums, limit):
    mx, mn = deque(), deque()
    l = best = 0
    for r, x in enumerate(nums):
        while mx and mx[-1] < x: mx.pop()     # giữ bản sao bằng nhau
        mx.append(x)
        while mn and mn[-1] > x: mn.pop()
        mn.append(x)
        while mx[0] - mn[0] > limit:
            if nums[l] == mx[0]: mx.popleft()
            if nums[l] == mn[0]: mn.popleft()
            l += 1
        best = max(best, r - l + 1)
    return best

rng = random.Random(3)
for _ in range(2000):
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 12))]
    L = rng.randint(0, 4)
    assert fixed_values_deque(a, L) == longest_brute(a, L)
print("Bản sửa (lưu giá trị, pop strict) khớp brute force trên 2000 test")

Bản sửa (lưu giá trị, pop strict) khớp brute force trên 2000 test


### Lỗi 4 — Heap mà quên xoá lười, đọc nhầm max/min "đã chết"

Nếu khi co cửa sổ ta chỉ tăng `l` một bước mà không kiểm tra chỉ số ở đỉnh heap, đỉnh có thể là phần tử đã rời cửa sổ. Ví dụ dưới chỉ tăng `l` một lần mỗi vòng (không lặp) nên vẫn kết thúc, nhưng cho kết quả sai.

In [26]:
def wrong_heap(nums, limit):
    maxh, minh = [], []
    l = best = 0
    for r, x in enumerate(nums):
        heapq.heappush(maxh, (-x, r))
        heapq.heappush(minh, (x, r))
        if -maxh[0][0] - minh[0][0] > limit:   # SAI: không dọn phần tử có chỉ số < l
            l += 1
        best = max(best, r - l + 1)
    return best

nums = [1, 9, 2, 3, 4, 5]
print("Sai :", wrong_heap(nums, 3), "| Đúng:", longest_deque(nums, 3))

Sai : 1 | Đúng: 4


**Sửa:** sau khi tăng `l`, luôn `while heap[0][1] < l: heappop(heap)` cho cả hai heap rồi **kiểm tra lại** điều kiện bằng `while` (cách 3).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"subarray dài nhất / ngắn nhất thoả điều kiện"** và điều kiện có tính **đơn điệu** (đúng với đoạn lớn ⇒ đúng với đoạn con) ⇒ nghĩ ngay đến **cửa sổ trượt hai con trỏ**.
- "Mọi cặp chênh ≤ X" ⇒ quy về **`max − min`**.
- Cần **max/min của cửa sổ trượt** ⇒ **deque đơn điệu** (LeetCode 239 Sliding Window Maximum là bài mẫu).
- Muốn code nhanh, ít lỗi trong phòng thi: `SortedList` (nếu có) hoặc 2 heap xoá lười.
- Lưu **chỉ số** trong deque/heap để biết phần tử đã rời cửa sổ chưa.
- Mẹo "cửa sổ không co lại": thay `while` bằng `if` khi co trái, cửa sổ chỉ trượt chứ không nhỏ đi; cuối cùng đáp án là `n − l`. Gọn nhưng khó giải thích hơn — dùng khi đã thạo.

In [27]:
def longest_nonshrinking(nums, limit):
    # cửa sổ không bao giờ nhỏ đi: dùng "if" thay "while"; đáp án = n - l
    maxq, minq = deque(), deque()
    l = 0
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] < x: maxq.pop()
        maxq.append(r)
        while minq and nums[minq[-1]] > x: minq.pop()
        minq.append(r)
        if nums[maxq[0]] - nums[minq[0]] > limit:
            l += 1
            if maxq[0] < l: maxq.popleft()
            if minq[0] < l: minq.popleft()
    return len(nums) - l

rng = random.Random(11)
for _ in range(1000):
    a = [rng.randint(1, 20) for _ in range(rng.randint(1, 25))]
    L = rng.randint(0, 10)
    assert longest_nonshrinking(a, L) == longest_brute(a, L)
print("Mẹo cửa sổ không co lại: khớp brute force")

Mẹo cửa sổ không co lại: khớp brute force


## 15. Biến thể của bài

1. **Đếm số subarray** có `max − min ≤ limit` (LeetCode 2762 Continuous Subarrays với limit 2): cùng cửa sổ, cộng `r − l + 1` mỗi bước.
2. **Sliding Window Maximum** (LeetCode 239): cửa sổ kích thước cố định `k`, chỉ cần một deque max.
3. **Trả về chính đoạn con** (vị trí bắt đầu/kết thúc) thay vì độ dài.
4. **Điều kiện khác dạng đơn điệu:** tổng ≤ S (với số dương), số phần tử khác nhau ≤ K (LeetCode 340), số số 0 lật ≤ K (LeetCode 1004).
5. **Max − min ≥ limit** (đoạn **ngắn nhất**): vẫn hai con trỏ nhưng co trái khi điều kiện **đúng**.

In [28]:
def count_subarrays(nums, limit):
    maxq, minq = deque(), deque()
    l = total = 0
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] < x: maxq.pop()
        maxq.append(r)
        while minq and nums[minq[-1]] > x: minq.pop()
        minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > limit:
            l += 1
            if maxq[0] < l: maxq.popleft()
            if minq[0] < l: minq.popleft()
        total += r - l + 1        # số đoạn hợp lệ kết thúc tại r
    return total

print(count_subarrays([5, 4, 2, 4], 2), "(LeetCode 2762 ví dụ 1 cho 8)")

8 (LeetCode 2762 ví dụ 1 cho 8)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Mọi điểm đầu, kéo dài + min/max chạy, `break` sớm | `O(n²)` | `O(1)` | Kiểm chứng, `n` nhỏ |
| 2. Sorted list | Cửa sổ trượt, list sắp xếp bằng `bisect` | `O(n·k)` (thực tế ổn) | `O(k)` | Muốn "multiset" dễ hiểu |
| 3. Hai heap | Cửa sổ trượt, xoá lười theo chỉ số | `O(n log n)` | `O(n)` | Quen heap, cần code chắc |
| 4. Hai deque đơn điệu | Ứng viên max/min, khấu hao | `O(n)` | `O(n)` | Lời giải tối ưu, phỏng vấn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output):** `longestSubarray([1,5,6,7,8,10,6,5,6], 4)` trả về bao nhiêu? Tự đoán trước khi xem đáp án.

**Bài 2 (tính tay):** với `nums = [10,1,2,4,7,2]`, `limit = 5`, hãy ghi danh sách `r − l + 1` (độ dài cửa sổ hợp lệ dài nhất kết thúc tại `r`) cho `r = 0..5`.

**Bài 3 (điền TODO):** hoàn thiện hàm `longest_todo` bên dưới (cách deque).

**Bài 4 (tự viết lại):** viết `count_valid(nums, limit)` đếm số subarray có `max − min ≤ limit` **mà không nhìn mục 15**.

**Bài 5 (mini challenge):** viết `leftmost_longest(nums, limit)` trả về cặp `(start, end)` của đoạn hợp lệ **dài nhất**, nếu nhiều đoạn cùng dài thì lấy đoạn có `start` **nhỏ nhất**.

In [29]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def longest_todo(nums, limit):
    maxq, minq = deque(), deque()
    l = best = 0
    for r, x in enumerate(nums):
        # TODO: bỏ các chỉ số ở cuối maxq có giá trị < x, rồi thêm r
        # TODO: bỏ các chỉ số ở cuối minq có giá trị > x, rồi thêm r
        # TODO: trong khi max - min > limit: tăng l, popleft nếu chỉ số đầu < l
        best = max(best, r - l + 1)
    return best

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Tìm các khúc mà max−min ≤ 4. (2) Thử khúc bắt đầu tại giá trị 5 hoặc 6. (3) Khúc `[6,7,8,10,6]`? max 10, min 6 → chênh 4.

**Bài 2:** (1) Dùng bảng trace ở mục 8. (2) Mỗi bước độ dài = `r − l + 1`. (3) `l` lần lượt là `0,1,1,1,2,2`.

**Bài 3:** (1) Giống cách 4. (2) Hai vòng `while ... pop()` trước khi `append(r)`. (3) Vòng co: `l += 1; if maxq[0] < l: maxq.popleft()` và tương tự `minq`.

**Bài 4:** (1) Mỗi `r` có `r − l + 1` đoạn hợp lệ kết thúc tại `r`. (2) Vì sao? Bổ đề 1 ở mục 10. (3) Cộng dồn thay vì lấy max.

**Bài 5:** (1) Chỉ cập nhật đáp án khi độ dài **lớn hơn hẳn** (`>`), không phải `>=`. (2) Lưu `start = l`, `end = r`. (3) Vì `r` tăng dần nên đoạn tìm thấy đầu tiên với độ dài lớn nhất có `end` nhỏ nhất, và với cùng độ dài thì `start = end − len + 1` cũng nhỏ nhất.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
ans1 = longest_deque([1, 5, 6, 7, 8, 10, 6, 5, 6], 4)
print("Bài 1:", ans1)
assert ans1 == longest_brute([1, 5, 6, 7, 8, 10, 6, 5, 6], 4) == 5

# Bài 2
def window_lengths(nums, limit):
    maxq, minq = deque(), deque()
    l, out = 0, []
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] < x: maxq.pop()
        maxq.append(r)
        while minq and nums[minq[-1]] > x: minq.pop()
        minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > limit:
            l += 1
            if maxq[0] < l: maxq.popleft()
            if minq[0] < l: minq.popleft()
        out.append(r - l + 1)
    return out

assert window_lengths([10, 1, 2, 4, 7, 2], 5) == [1, 1, 2, 3, 3, 4]
print("Bài 2:", window_lengths([10, 1, 2, 4, 7, 2], 5))

Bài 1: 5
Bài 2: [1, 1, 2, 3, 3, 4]


In [31]:
# Bài 3 — bản hoàn chỉnh
def longest_todo_sol(nums, limit):
    maxq, minq = deque(), deque()
    l = best = 0
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] < x:
            maxq.pop()
        maxq.append(r)
        while minq and nums[minq[-1]] > x:
            minq.pop()
        minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > limit:
            l += 1
            if maxq[0] < l:
                maxq.popleft()
            if minq[0] < l:
                minq.popleft()
        best = max(best, r - l + 1)
    return best

# Bài 4
def count_valid(nums, limit):
    return sum(window_lengths(nums, limit))

def count_valid_brute(nums, limit):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i, n)
               if max(nums[i:j + 1]) - min(nums[i:j + 1]) <= limit)

# Bài 5
def leftmost_longest(nums, limit):
    maxq, minq = deque(), deque()
    l, best, ans = 0, 0, (0, 0)
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] < x: maxq.pop()
        maxq.append(r)
        while minq and nums[minq[-1]] > x: minq.pop()
        minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > limit:
            l += 1
            if maxq[0] < l: maxq.popleft()
            if minq[0] < l: minq.popleft()
        if r - l + 1 > best:
            best, ans = r - l + 1, (l, r)
    return ans

def leftmost_longest_brute(nums, limit):
    n, best, ans = len(nums), 0, (0, 0)
    for i in range(n):
        for j in range(i, n):
            seg = nums[i:j + 1]
            if max(seg) - min(seg) <= limit and j - i + 1 > best:
                best, ans = j - i + 1, (i, j)
    return ans

assert leftmost_longest([8, 2, 4, 7], 4) == (1, 2)
rng = random.Random(19)
for _ in range(600):
    a = [rng.randint(1, 9) for _ in range(rng.randint(1, 14))]
    L = rng.randint(0, 6)
    assert longest_todo_sol(a, L) == longest_brute(a, L)
    assert count_valid(a, L) == count_valid_brute(a, L)
    assert leftmost_longest(a, L) == leftmost_longest_brute(a, L)
print("Đáp án bài 3, 4, 5: OK (600 test ngẫu nhiên)")

Đáp án bài 3, 4, 5: OK (600 test ngẫu nhiên)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template cửa sổ trượt + deque đơn điệu (max/min):**

```python
maxq, minq = deque(), deque()          # lưu CHỈ SỐ
l = best = 0
for r, x in enumerate(nums):
    while maxq and nums[maxq[-1]] < x: maxq.pop()
    maxq.append(r)
    while minq and nums[minq[-1]] > x: minq.pop()
    minq.append(r)
    while nums[maxq[0]] - nums[minq[0]] > limit:   # cửa sổ sai → co trái
        l += 1
        if maxq[0] < l: maxq.popleft()
        if minq[0] < l: minq.popleft()
    best = max(best, r - l + 1)
```

**Checklist:**
- [ ] Đã quy "mọi cặp" về `max − min`, không chỉ so phần tử kề nhau.
- [ ] Deque lưu **chỉ số**; pop cuối với dấu **strict** (`<` cho max, `>` cho min).
- [ ] Co trái bằng `while` (hoặc dùng mẹo `n − l` một cách có ý thức).
- [ ] Độ dài cửa sổ là `r − l + 1`.
- [ ] Đã thử: 1 phần tử, `limit = 0`, phần tử trùng nhau, `limit` cực lớn, mảng tăng/giảm dần.